# FL Server - Federated Learning Aggregation Server

Notebook ini menjalankan **FL Server** yang bertugas:
1. Menyimpan dan mendistribusikan **Global Model**
2. Menerima **weights** dari setiap client setelah local training
3. Melakukan **FedAvg aggregation**
4. Mengevaluasi model pada **test data**

## Cara Menjalankan (Google Colab):

### Persiapan (sekali saja):
1. Daftar akun ngrok GRATIS di: https://dashboard.ngrok.com/signup
2. Copy authtoken dari: https://dashboard.ngrok.com/get-started/your-authtoken

### Menjalankan Server:
1. Paste authtoken di cell **Ngrok Authtoken**
2. Jalankan semua cell sampai server running
3. **COPY URL ngrok** yang muncul (contoh: `https://xxxx.ngrok-free.app`)
4. Buka `client_1.ipynb` dan `client_2.ipynb` di tab browser baru
5. **PASTE URL ngrok** di cell Configuration kedua client
6. Jalankan kedua client notebook

---

## 1. Install Dependencies

In [1]:
# Install dependencies
!pip install flask efficientnet_pytorch pyngrok -q

  Preparing metadata (setup.py) ... done


## 2. Download Dataset dari GitHub

In [2]:
# ========================================
# DOWNLOAD DATASET DARI GITHUB (SPARSE CHECKOUT)
# ========================================
# Server hanya membutuhkan test data untuk evaluasi

import os

# URL GitHub repository (GANTI DENGAN URL REPO ANDA)
GITHUB_REPO_URL = 'https://github.com/nashuhainsani/test'
DATA_FOLDER = 'day-3/data/test'  # Server hanya perlu test data
LOCAL_DIR = 'workshop-data'

# Download hanya folder data menggunakan sparse checkout
if not os.path.exists(LOCAL_DIR):
    print('Downloading test dataset...')
    !git clone --filter=blob:none --sparse {GITHUB_REPO_URL} {LOCAL_DIR}
    %cd {LOCAL_DIR}
    !git sparse-checkout set {DATA_FOLDER}
    %cd ..
    print('Download selesai!')
else:
    print(f'Data sudah ada di folder {LOCAL_DIR}/')

# Set BASE_DIR ke folder yang berisi data
BASE_DIR = os.path.join(LOCAL_DIR, 'day-3')
print(f'Data directory: {BASE_DIR}/')
print(f'Test data: {os.path.join(BASE_DIR, "data/test")}')

Cloning into 'workshop-data'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 23 (delta 2), reused 7 (delta 2), pack-reused 13 (from 1)
Receiving objects: 100% (23/23), 39.49 KiB | 9.87 MiB/s, done.
Resolving deltas: 100% (2/2), done.
remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 1 (from 1)
Receiving objects: 100% (2/2), 250 bytes | 250.00 KiB/s, done.
/content/workshop-data
remote: Enumerating objects: 251, done.
remote: Total 251 (delta 0), reused 0 (delta 0), pack-reused 251 (from 1)
Receiving objects: 100% (251/251), 106.49 MiB | 18.16 MiB/s, done.
Updating files: 100% (253/253), done.
/content
Download selesai!
Data directory: workshop-data/day-3/
Test data: workshop-data/day-3/data/test


## 3. Import Libraries

In [3]:
import os
import io
import copy
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from datetime import datetime
from flask import Flask, request, jsonify, send_file
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from efficientnet_pytorch import EfficientNet
import threading
import gzip
import base64
print('Libraries imported!')

Libraries imported!


## 4. Configuration & Ngrok Setup

### Setup Ngrok (WAJIB untuk Google Colab)

Ngrok membuat public URL agar client bisa connect ke server dari notebook berbeda.

**Langkah mendapatkan Authtoken (GRATIS):**
1. Buka https://dashboard.ngrok.com/signup dan daftar akun gratis
2. Setelah login, buka https://dashboard.ngrok.com/get-started/your-authtoken
3. Copy authtoken dan paste di cell berikutnya

In [4]:
# ========================================
# NGROK AUTHTOKEN (WAJIB DIISI!)
# ========================================
# Daftar gratis di: https://dashboard.ngrok.com/signup
# Copy authtoken dari: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_AUTHTOKEN = '3HaIcL8M3igynCnZDdnXkUJ9Cns_7M6hyFYhUNCsYCNtMa17'

# Validasi
if 'PASTE' in NGROK_AUTHTOKEN or len(NGROK_AUTHTOKEN) < 20:
    print('='*60)
    print('ERROR: NGROK_AUTHTOKEN belum diisi!')
    print('')
    print('1. Daftar gratis di: https://dashboard.ngrok.com/signup')
    print('2. Copy authtoken dari: https://dashboard.ngrok.com/get-started/your-authtoken')
    print('3. Paste di variabel NGROK_AUTHTOKEN di atas')
    print('='*60)
else:
    print('Authtoken OK!')

Authtoken OK!


In [5]:
# ========================================
# START NGROK TUNNEL
# ========================================
from pyngrok import ngrok, conf
import os

# Set authtoken
conf.get_default().auth_token = NGROK_AUTHTOKEN

# Kill any existing ngrok processes
ngrok.kill()

# Server config
PORT = 5000

# Start ngrok tunnel
public_url = ngrok.connect(PORT).public_url
SERVER_URL = public_url

print('='*60)
print('NGROK TUNNEL CREATED!')
print('='*60)
print(f'\n>>> PUBLIC SERVER URL: {SERVER_URL} <<<\n')
print('COPY URL ini ke notebook client_1 dan client_2!')
print('='*60)

NGROK TUNNEL CREATED!

>>> PUBLIC SERVER URL: https://hunger-ninth-nebula.ngrok-free.dev <<<

COPY URL ini ke notebook client_1 dan client_2!


In [6]:
# Model config
N_CLASSES = 2
CLASS_NAMES = ['Normal', 'Abnormal']

# Paths (dari GitHub clone)
MODEL_DIR = 'models'
GLOBAL_MODEL_PATH = os.path.join(MODEL_DIR, 'global_model.pth')
BEST_MODEL_PATH = os.path.join(MODEL_DIR, 'best_model.pth')
TEST_DATA_DIR = os.path.join(BASE_DIR, 'data/test')

os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Server URL: {SERVER_URL}')
print(f'Test data: {TEST_DATA_DIR}')

Server URL: https://hunger-ninth-nebula.ngrok-free.dev
Test data: workshop-data/day-3/data/test


## 5. Model Architecture (sama dengan client)

In [7]:
class EfficientNetB0(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.model = EfficientNet.from_pretrained('efficientnet-b0')
        self.num_ftrs = self.model._fc.in_features
        self.model._fc = nn.Linear(self.num_ftrs, n_classes)
        self.projector = nn.Sequential(
            nn.Linear(self.num_ftrs, self.num_ftrs),
            nn.Linear(self.num_ftrs, 1024)
        )

    def forward(self, x, project=False):
        features = self.model.extract_features(x)
        features = self.model._avg_pooling(features)
        features = features.flatten(start_dim=1)
        if project:
            proj = self.projector(features)
            out = self.model._dropout(features)
            out = self.model._fc(out)
            return proj, out
        else:
            out = self.model._dropout(features)
            out = self.model._fc(out)
            return out, out

print('Model architecture defined!')

Model architecture defined!


## 6. FL State & Helper Functions

In [8]:
# FL State
fl_state = {
    'current_round': 0,
    'registered_clients': {},
    'received_weights': {},
    'client_data_sizes': {},
    'is_aggregating': False,
    'expected_clients': 2,
    'best_bacc': 0.0,
    'best_round': -1,
    'evaluation_history': []
}

chunked_uploads = {}

def fedavg(weights_list, data_sizes):
    '''FedAvg: Weighted average berdasarkan jumlah data'''
    total = sum(data_sizes)
    w_avg = copy.deepcopy(weights_list[0])
    for key in w_avg.keys():
        w_avg[key] = w_avg[key] * (data_sizes[0] / total)
        for i in range(1, len(weights_list)):
            w_avg[key] += weights_list[i][key] * (data_sizes[i] / total)
    return w_avg

def save_global_model(state_dict):
    torch.save(state_dict, GLOBAL_MODEL_PATH)
    print(f'Global model saved to {GLOBAL_MODEL_PATH}')

print('FL state and helpers initialized!')

FL state and helpers initialized!


## 7. Test Dataset & Evaluation

In [9]:
class TestDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.samples = []
        self.transform = transform
        csv_path = os.path.join(data_dir, 'labels.csv')
        images_dir = os.path.join(data_dir, 'images')
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            img_path = os.path.join(images_dir, row['filename'])
            if os.path.exists(img_path):
                self.samples.append((img_path, int(row['label'])))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, label

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def evaluate_model(state_dict):
    '''Evaluate global model on test data'''
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = EfficientNetB0(N_CLASSES).to(device)
    model.load_state_dict(state_dict)
    model.eval()

    test_dataset = TestDataset(TEST_DATA_DIR, test_transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            _, logits = model(images.to(device))
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    all_preds, all_labels = np.array(all_preds), np.array(all_labels)
    acc = np.mean(all_preds == all_labels)

    # Balanced accuracy
    recall_per_class = []
    for c in range(N_CLASSES):
        mask = all_labels == c
        if np.sum(mask) > 0:
            recall_per_class.append(np.sum((all_preds == c) & mask) / np.sum(mask))
    bacc = np.mean(recall_per_class)

    return {'accuracy': acc, 'balanced_accuracy': bacc, 'test_size': len(test_dataset)}

print('Test dataset and evaluation ready!')

Test dataset and evaluation ready!


## 8. Initialize Global Model

In [10]:
# Initialize global model from pretrained
print('Initializing global model from pretrained...')
model = EfficientNetB0(N_CLASSES)
save_global_model(model.state_dict())
print('Global model initialized!')

Initializing global model from pretrained...
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 241MB/s]


Loaded pretrained weights for efficientnet-b0
Global model saved to models/global_model.pth
Global model initialized!


## 9. Flask Server App

In [11]:
app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'healthy', 'round': fl_state['current_round']})

@app.route('/register', methods=['POST'])
def register():
    data = request.json
    client_id = data.get('client_id')
    data_size = data.get('data_size', 0)
    fl_state['registered_clients'][client_id] = {'data_size': data_size}
    print(f'Client {client_id} registered with {data_size} samples')
    return jsonify({'status': 'registered', 'current_round': fl_state['current_round']})

@app.route('/model/download')
def download_model():
    if not os.path.exists(GLOBAL_MODEL_PATH):
        return jsonify({'error': 'No model available'}), 404
    client_id = request.args.get('client_id', 'unknown')
    print(f'Client {client_id} downloading model (round {fl_state["current_round"]})')
    return send_file(GLOBAL_MODEL_PATH, mimetype='application/octet-stream')

@app.route('/status')
def status():
    return jsonify({
        'current_round': fl_state['current_round'],
        'registered_clients': list(fl_state['registered_clients'].keys()),
        'weights_received_count': len(fl_state['received_weights']),
        'expected_clients': fl_state['expected_clients'],
        'best_bacc': fl_state['best_bacc'],
        'best_round': fl_state['best_round']
    })

print('Basic endpoints defined!')

Basic endpoints defined!


## 10. Chunked Upload & Aggregation Endpoints

In [12]:
@app.route('/model/upload_chunk', methods=['POST'])
def upload_chunk():
    data = request.json
    upload_id = data.get('upload_id')
    chunk_idx = data.get('chunk_idx')
    chunk_data = data.get('chunk_data')
    total_chunks = data.get('total_chunks')

    if upload_id not in chunked_uploads:
        chunked_uploads[upload_id] = {
            'client_id': data.get('client_id'),
            'total_chunks': total_chunks,
            'chunks': {},
            'data_size': data.get('data_size', 0),
            'round': data.get('round', 0)
        }
    chunked_uploads[upload_id]['chunks'][chunk_idx] = chunk_data
    return jsonify({'status': 'chunk_received', 'chunk_idx': chunk_idx})

@app.route('/model/upload_complete', methods=['POST'])
def upload_complete():
    data = request.json
    upload_id = data.get('upload_id')
    client_id = data.get('client_id')
    data_size = data.get('data_size', 0)

    if upload_id not in chunked_uploads:
        return jsonify({'error': 'Upload not found'}), 404

    upload_data = chunked_uploads[upload_id]
    total_chunks = upload_data['total_chunks']

    # Reassemble
    encoded_data = ''.join(upload_data['chunks'][i] for i in range(total_chunks))
    compressed_data = base64.b64decode(encoded_data)
    weights_data = gzip.decompress(compressed_data)
    weights = torch.load(io.BytesIO(weights_data), map_location='cpu')

    del chunked_uploads[upload_id]

    fl_state['received_weights'][client_id] = weights
    fl_state['client_data_sizes'][client_id] = data_size

    num_received = len(fl_state['received_weights'])
    print(f'Weights received from {client_id} ({num_received}/{fl_state["expected_clients"]})')

    # Auto aggregate if all clients uploaded
    result = None
    if num_received >= fl_state['expected_clients']:
        result = do_aggregation()

    response = {'status': 'received', 'total_received': num_received}
    if result:
        response['aggregation'] = result
        response['new_round'] = fl_state['current_round']
    return jsonify(response)

def do_aggregation():
    '''Perform FedAvg aggregation'''
    if fl_state['is_aggregating']:
        return None
    fl_state['is_aggregating'] = True

    try:
        weights_list = list(fl_state['received_weights'].values())
        data_sizes = [fl_state['client_data_sizes'][cid] for cid in fl_state['received_weights']]
        client_ids = list(fl_state['received_weights'].keys())

        print('='*50)
        print(f'FEDAVG AGGREGATION - Round {fl_state["current_round"]}')
        print(f'Clients: {client_ids}')
        print(f'Data sizes: {data_sizes}')

        aggregated = fedavg(weights_list, data_sizes)
        save_global_model(aggregated)

        # Evaluate
        eval_result = evaluate_model(aggregated)
        bacc = eval_result['balanced_accuracy']
        print(f'Test Accuracy: {eval_result["accuracy"]*100:.2f}%')
        print(f'Test Balanced Accuracy: {bacc*100:.2f}%')

        if bacc > fl_state['best_bacc']:
            fl_state['best_bacc'] = bacc
            fl_state['best_round'] = fl_state['current_round']
            torch.save(aggregated, BEST_MODEL_PATH)
            print(f'NEW BEST MODEL! BACC: {bacc*100:.2f}%')

        fl_state['evaluation_history'].append({
            'round': fl_state['current_round'],
            'bacc': bacc
        })

        old_round = fl_state['current_round']
        fl_state['current_round'] += 1
        fl_state['received_weights'] = {}
        fl_state['client_data_sizes'] = {}

        print(f'Round {old_round} -> {fl_state["current_round"]}')
        print('='*50)

        return {'status': 'aggregated', 'bacc': bacc}
    finally:
        fl_state['is_aggregating'] = False

print('Upload and aggregation endpoints defined!')

Upload and aggregation endpoints defined!


## 11. Start Server

**PENTING**: Jalankan cell ini, lalu buka notebook client_1 dan client_2!

In [ ]:
print('='*50)
print('STARTING FL SERVER')
print('='*50)
print(f'Public URL: {SERVER_URL}')
print(f'Expected clients: {fl_state["expected_clients"]}')
print('')
print('>>> COPY URL DI ATAS KE NOTEBOOK CLIENT! <<<')
print('')
print('Setelah server running, jalankan:')
print('  1. client_1.ipynb')
print('  2. client_2.ipynb')
print('='*50)

# Run Flask (use_reloader=False for Jupyter)
app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

STARTING FL SERVER
Public URL: https://hunger-ninth-nebula.ngrok-free.dev
Expected clients: 2

>>> COPY URL DI ATAS KE NOTEBOOK CLIENT! <<<

Setelah server running, jalankan:
  1. client_1.ipynb
  2. client_2.ipynb
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:31:23] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:31:24] "POST /register HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:31:24] "GET /model/download?client_id=client_1 HTTP/1.1" 200 -


Client client_1 registered with 425 samples
Client client_1 downloading model (round 0)


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:31:40] "GET /health HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:31:40] "POST /register HTTP/1.1" 200 -


Client client_2 registered with 595 samples


...

INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:30] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:35] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:40] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:45] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:50] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:44:56] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:01] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:06] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:11] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:16] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:22] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:45:27] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026

Weights received from client_2 (2/2)
FEDAVG AGGREGATION - Round 2
Clients: ['client_1', 'client_2']
Data sizes: [425, 595]
Global model saved to models/global_model.pth
Loaded pretrained weights for efficientnet-b0


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:08] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:14] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:19] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:24] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:29] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:34] "GET /status HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:38] "POST /model/upload_complete HTTP/1.1" 200 -


Test Accuracy: 28.40%
Test Balanced Accuracy: 28.40%
Round 2 -> 3


INFO:werkzeug:127.0.0.1 - - [11/Aug/2026 11:46:40] "GET /status HTTP/1.1" 200 -
